In [26]:
import numpy as np
from sklearn.decomposition import PCA
import pandas as pd
from sklearn.preprocessing import StandardScaler
import re
import plotly.express as px
import dill
from scipy.interpolate import RBFInterpolator
from SALib.sample import latin

In [27]:
def get_n(s):
    match = re.search(r'\d+$', s)
    if match:
        return int(match.group())
    else:
        return None

In [28]:
headers = ['sim_name', 'alpha', 'beta', 'vd', 'sg', 'D']
PDE_names = pd.read_csv('/Users/laurenyan/bmxfungi_surrogate/pareto_mean.csv', header=None, names=headers)['sim_name']
PDE_mn_df = pd.read_csv('/Users/laurenyan/bmxfungi_surrogate/pareto_mean.csv', header=None, names=headers)
PDE_mn_df['rho_sat'] = 2 * PDE_mn_df['alpha'] / PDE_mn_df['beta']
PDE_mn_df['c'] = PDE_mn_df['vd'] + 2 * np.sqrt(PDE_mn_df['D'] * PDE_mn_df['alpha'])

PDE_scaler = StandardScaler()
PDE_scaler.fit(PDE_mn_df[['alpha', 'beta', 'vd', 'sg', 'D']])
PDE_mn_df[['alpha_stdd', 'beta_stdd', 'vd_stdd', 'sg_stdd', 'D_stdd']] = PDE_scaler.transform(PDE_mn_df[['alpha', 'beta', 'vd', 'sg', 'D']])

In [29]:
params = ['max_l', 'split_l', 'max_r', 's_p', 'f_p', 'f_separation', 'W_A']
param_names = ['chem_species.max_seg_length', 'chem_species.seg_split_length',
               'chem_species.max_seg_radius', 'chem_species.splitting_probability',
               'chem_species.fusion_probability', 'chem_species.max_fusion_separation',
               'cell_force.fungi_boundary_width'] 

ABM_256_inputs = pd.read_csv('/Users/laurenyan/bmxfungi_surrogate/256_inputs.csv', index_col= 0)

ABM_sample = np.zeros((len(PDE_names), 7))
j = 0
sim_names = PDE_names.to_list()
for sim in sim_names:
    n = get_n(sim)
    sim_row = ABM_256_inputs.iloc[n]
    i = 0
    for name in param_names:
        ABM_sample[j, i] = sim_row[name]
        i += 1
    j += 1

In [30]:
ABM_df = pd.DataFrame(ABM_sample, columns=params)
ABM_df.insert(0, 'sim_name', sim_names)

ABM_scaler = StandardScaler()
ABM_scaler.fit(ABM_df[['max_l', 'split_l', 'max_r', 's_p', 'f_p', 'f_separation', 'W_A']])
ABM_df[['max_l_stdd', 'split_l_stdd', 'max_r_stdd', 's_p_stdd', 'f_p_stdd', 'f_separation_stdd', 'W_A_stdd']] = ABM_scaler.transform(ABM_df[['max_l', 'split_l', 'max_r', 's_p', 'f_p', 'f_separation', 'W_A']])

In [31]:
in_out = ABM_df.merge(PDE_mn_df, on = 'sim_name', how ='inner')

In [32]:
in_out

,sim_name,max_l,split_l,max_r,s_p,f_p,f_separation,W_A,max_l_stdd,split_l_stdd,...,vd,sg,D,rho_sat,c,alpha_stdd,beta_stdd,vd_stdd,sg_stdd,D_stdd
0,output_fungi_104,0.004208,0.003929,0.000332,0.20,0.01388,0.000831,0.000142,0.699770,0.938180,...,0.011279,0.059516,0.000182,58.367777,0.023155,-0.953450,-0.515546,-0.429305,-0.849399,-0.072915
1,output_fungi_131,0.003251,0.002546,0.000341,0.19,0.00677,0.000514,0.000101,-0.328460,-0.570120,...,0.005930,0.083790,0.000077,381.621753,0.013363,-1.081738,-1.058923,-0.859755,-0.063260,-0.158585
2,output_fungi_105,0.004048,0.003493,0.000179,0.11,0.00523,0.000778,0.000062,0.527861,0.462678,...,0.015058,0.073960,0.000166,36.044930,0.027676,-0.563045,0.115787,-0.125207,-0.381609,-0.085545
3,output_fungi_132,0.005245,0.004936,0.000159,0.25,0.01391,0.000871,0.000059,1.813955,2.036414,...,0.024576,0.096120,0.000051,13.804747,0.033141,0.481569,3.825401,0.640809,0.336071,-0.180093
4,output_fungi_133,0.004657,0.003994,0.000239,0.22,0.01369,0.000634,0.000122,1.182190,1.009069,...,0.029366,0.102816,0.000052,19.153860,0.038645,0.911248,2.932423,1.026239,0.552927,-0.178796
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
148,output_fungi_8,0.004351,0.003995,0.000257,0.10,0.00664,0.000491,0.000115,0.853414,1.010159,...,0.002719,0.063358,0.000229,30.734124,0.013506,-1.527988,-0.360000,-1.118176,-0.724978,-0.034577
149,output_fungi_92,0.004768,0.004513,0.000186,0.21,0.00692,0.000656,0.000106,1.301451,1.575090,...,0.001779,0.053653,0.000079,46.167747,0.008110,-1.535792,-0.627250,-1.193787,-1.039305,-0.156800
150,output_fungi_93,0.005141,0.004485,0.000227,0.23,0.00544,0.000839,0.000068,1.702214,1.544553,...,0.005885,0.034229,0.000118,17.305946,0.017374,-0.208478,1.937523,-0.863387,-1.668389,-0.125403
151,output_fungi_94,0.003139,0.002777,0.000273,0.18,0.01259,0.000483,0.000123,-0.448796,-0.318191,...,0.007457,0.068135,0.000071,129.717267,0.014979,-0.918689,-0.857639,-0.736878,-0.570274,-0.163244


In [33]:
pca = PCA(n_components=3)
components = pca.fit_transform(in_out[['max_l_stdd', 'split_l_stdd', 'max_r_stdd', 's_p_stdd', 'f_p_stdd', 'f_separation_stdd', 'W_A_stdd', 'alpha_stdd', 'beta_stdd', 'vd_stdd', 'sg_stdd', 'D_stdd']])
labels = {
    str(i): f"PC {i+1} ({var:.1f}%)"
    for i, var in enumerate(pca.explained_variance_ratio_ * 100)
}

fig = px.scatter_matrix(
    components,
    labels=labels,
    dimensions=range(3),
    color=np.log(PDE_mn_df['rho_sat'])
)
fig.update_traces(diagonal_visible=False)
import plotly.io as pio
pio.renderers.default = 'iframe'
fig.update_layout(width=1000, height=700)
fig.show()

In [36]:
components

array([[ 1.19275110e+00, -1.54373210e+00],
       [-8.12373460e-01, -2.01089948e+00],
       [ 9.31244721e-01, -4.07513468e-02],
       [ 3.63979434e+00,  3.26856333e+00],
       [ 1.78444228e+00,  2.27814602e+00],
       [-1.80612183e+00, -5.21978897e-01],
       [-9.78327133e-01, -1.20351606e+00],
       [ 2.50286168e+00, -1.99082707e-01],
       [ 1.00937821e+00,  4.34257524e-01],
       [-3.51783919e-01, -2.86895473e-01],
       [-1.74878670e+00, -1.42292367e+00],
       [-7.35624744e-02, -8.86503004e-01],
       [ 4.31398228e-02, -3.04633491e-01],
       [ 1.38816871e+00,  1.43230821e+00],
       [-1.87445842e+00,  1.17388725e-01],
       [-3.78847385e-02,  3.55272256e-01],
       [ 2.45578727e-01, -1.33991050e+00],
       [ 1.78871966e+00, -1.24081556e+00],
       [ 1.68236007e+00,  1.87100825e+00],
       [-5.49669779e-01,  2.70918225e-01],
       [ 2.54948432e+00,  1.67881377e-01],
       [ 1.87011530e+00,  2.05686067e+00],
       [-1.42233335e+00, -4.71768818e-01],
       [-7.

In [34]:
pca = PCA(n_components=2)
components = pca.fit_transform(in_out[['max_l_stdd', 'split_l_stdd', 'max_r_stdd', 's_p_stdd', 'f_p_stdd', 'f_separation_stdd', 'W_A_stdd', 'alpha_stdd', 'beta_stdd', 'vd_stdd', 'sg_stdd', 'D_stdd']])
loadings = pca.components_.T * np.sqrt(pca.explained_variance_)
fig = px.scatter(components, x=0, y=1, color=np.log(in_out['rho_sat']))
features = ['max_l', 'split_l', 'max_r', 's_p', 'f_p', 'f_separation', 'W_A', 'alpha', 'beta', 'vd', 'sg', 'D']
for i, feature in enumerate(features):
    fig.add_annotation(
        ax=0, ay=0,
        axref="x", ayref="y",
        x=loadings[i, 0] * 5,
        y=loadings[i, 1] * 5,
        showarrow=True,
        arrowsize=2,
        arrowhead=2,
        xanchor="right",
        yanchor="top"
    )
    fig.add_annotation(
        x=loadings[i, 0] * 5,
        y=loadings[i, 1] * 5,
        ax=0, ay=0,
        xanchor="center",
        yanchor="bottom",
        text=feature,
        yshift=5,
    )
fig.show()

In [18]:
# ['max_l', 'split_l', 'max_r', 's_p', 'f_p', 'f_separation', 'W_A']
a = 'max_r'
pca = PCA(n_components=2)
components = pca.fit_transform(in_out[[a + '_stdd', 'alpha_stdd', 'beta_stdd', 'vd_stdd', 'sg_stdd', 'D_stdd']])
loadings = pca.components_.T * np.sqrt(pca.explained_variance_)
fig = px.scatter(components, x=0, y=1, color=np.log(in_out['c']))
features = [a, 'alpha', 'beta', 'vd', 'sg', 'D']
for i, feature in enumerate(features):
    fig.add_annotation(
        ax=0, ay=0,
        axref="x", ayref="y",
        x=loadings[i, 0] * 5,
        y=loadings[i, 1] * 5,
        showarrow=True,
        arrowsize=1,
        arrowhead=1,
        xanchor="right",
        yanchor="top"
    )
    fig.add_annotation(
        x=loadings[i, 0] * 5,
        y=loadings[i, 1] * 5,
        ax=0, ay=0,
        xanchor="center",
        yanchor="bottom",
        text=feature,
        yshift=5,
    )
fig.show()

In [19]:
pca = PCA(n_components=2)
components = pca.fit_transform(in_out[['alpha_stdd', 'beta_stdd', 'vd_stdd', 'sg_stdd', 'D_stdd']])
loadings = pca.components_.T * np.sqrt(pca.explained_variance_)
fig = px.scatter(components, x=0, y=1, color=np.log(in_out['rho_sat']))
features = ['alpha', 'beta', 'vd', 'sg', 'D']
for i, feature in enumerate(features):
    fig.add_annotation(
        ax=0, ay=0,
        axref="x", ayref="y",
        x=loadings[i, 0] * 5,
        y=loadings[i, 1] * 5,
        showarrow=True,
        arrowsize=2,
        arrowhead=2,
        xanchor="right",
        yanchor="top"
    )
    fig.add_annotation(
        x=loadings[i, 0] * 5,
        y=loadings[i, 1] * 5,
        ax=0, ay=0,
        xanchor="center",
        yanchor="bottom",
        text=feature,
        yshift=5,
    )
fig.show()

In [20]:
PDE_srgte_df = pd.read_csv('PDE_in_mn.csv', index_col = 0)
PDE_scaler = StandardScaler()
PDE_scaler.fit(PDE_srgte_df[['alpha', 'beta', 'vd', 'sg', 'D']])
PDE_srgte_df[['alpha_stdd', 'beta_stdd', 'vd_stdd', 'sg_stdd', 'D_stdd']] = PDE_scaler.transform(PDE_srgte_df[['alpha', 'beta', 'vd', 'sg', 'D']])
ABM__srgte_df = pd.read_csv('ABM_in.csv', index_col = 0)
ABM_scaler = StandardScaler()
ABM_scaler.fit(ABM__srgte_df[['max_l', 'split_l', 'max_r', 's_p', 'f_p', 'f_separation', 'W_A']])
ABM__srgte_df[['max_l_stdd', 'split_l_stdd', 'max_r_stdd', 's_p_stdd', 'f_p_stdd', 'f_separation_stdd', 'W_A_stdd']] = ABM_scaler.transform(ABM__srgte_df[['max_l', 'split_l', 'max_r', 's_p', 'f_p', 'f_separation', 'W_A']])
in_out_srgte = ABM__srgte_df.merge(PDE_srgte_df, left_index=True, right_index=True)

In [21]:
in_out_srgte

,max_l,split_l,max_r,s_p,f_p,f_separation,W_A,max_l_stdd,split_l_stdd,max_r_stdd,...,vd,sg,D,rho_sat,c,alpha_stdd,beta_stdd,vd_stdd,sg_stdd,D_stdd
0,0.004155,0.003782,0.000170,0.174475,0.007377,0.000506,0.000068,1.661525,1.771938,-1.373774,...,0.015039,0.062194,0.000412,30.482130,0.035464,-0.539597,1.686643,-0.434434,-1.410497,0.771778
1,0.002972,0.002577,0.000293,0.178146,0.008555,0.000679,0.000064,-0.351580,-0.258298,1.604569,...,0.013945,0.071680,0.000197,52.211812,0.028518,-0.262922,0.162673,-0.736080,-0.827472,-1.022670
2,0.003784,0.003245,0.000206,0.129309,0.009257,0.000474,0.000116,1.031004,0.867837,-0.489460,...,0.013312,0.076270,0.000229,43.761744,0.026739,-1.532467,-0.152975,-0.910703,-0.545333,-0.758453
3,0.003179,0.002828,0.000226,0.199519,0.006982,0.000383,0.000079,0.000558,0.164748,-0.008838,...,0.015727,0.078221,0.000540,110.473163,0.040657,0.063434,-1.067937,-0.244498,-0.425413,1.847368
4,0.004098,0.003506,0.000246,0.176016,0.006622,0.000389,0.000063,1.564816,1.306990,0.461975,...,0.013164,0.082998,0.000414,53.714327,0.034699,-0.071476,0.190884,-0.951593,-0.131805,0.792085
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1995,0.003546,0.003231,0.000157,0.125148,0.007472,0.000484,0.000078,0.625972,0.844806,-1.679079,...,0.014705,0.065539,0.000290,41.276944,0.030191,-1.364826,0.089167,-0.526380,-1.204923,-0.243349
1996,0.003788,0.003324,0.000263,0.205028,0.011926,0.000551,0.000098,1.037535,1.001266,0.867132,...,0.014826,0.077296,0.000328,42.865587,0.032711,-0.708995,0.419460,-0.493165,-0.482307,0.072219
1997,0.003845,0.003394,0.000228,0.209169,0.010888,0.000633,0.000119,1.134347,1.119486,0.037833,...,0.017406,0.072385,0.000301,32.839698,0.034998,-0.483462,1.445582,0.218669,-0.784150,-0.150008
1998,0.003374,0.002913,0.000207,0.145782,0.010489,0.000568,0.000076,0.332830,0.308907,-0.464346,...,0.015208,0.075112,0.000251,41.575418,0.030836,-0.713757,0.501467,-0.387613,-0.616506,-0.574849


In [22]:
pca = PCA(n_components=2)
components = pca.fit_transform(in_out_srgte[['alpha_stdd', 'beta_stdd', 'vd_stdd', 'sg_stdd', 'D_stdd']])
loadings = pca.components_.T * np.sqrt(pca.explained_variance_)
fig = px.scatter(components, x=0, y=1, color=np.log(in_out_srgte['rho_sat']))
features = ['alpha', 'beta', 'vd', 'sg', 'D']
for i, feature in enumerate(features):
    fig.add_annotation(
        ax=0, ay=0,
        axref="x", ayref="y",
        x=loadings[i, 0] * 5,
        y=loadings[i, 1] * 5,
        showarrow=True,
        arrowsize=2,
        arrowhead=2,
        xanchor="right",
        yanchor="top"
    )
    fig.add_annotation(
        x=loadings[i, 0] * 5,
        y=loadings[i, 1] * 5,
        ax=0, ay=0,
        xanchor="center",
        yanchor="bottom",
        text=feature,
        yshift=5,
    )
fig.show()

In [25]:
# ['max_l', 'split_l', 'max_r', 's_p', 'f_p', 'f_separation', 'W_A']
a = 'max_r'
pca = PCA(n_components=2)
components = pca.fit_transform(in_out_srgte[[a + '_stdd', 'alpha_stdd', 'beta_stdd', 'vd_stdd', 'sg_stdd', 'D_stdd']])
loadings = pca.components_.T * np.sqrt(pca.explained_variance_)
fig = px.scatter(components, x=0, y=1, color=in_out_srgte['c'])
features = [a, 'alpha', 'beta', 'vd', 'sg', 'D']
for i, feature in enumerate(features):
    fig.add_annotation(
        ax=0, ay=0,
        axref="x", ayref="y",
        x=loadings[i, 0] * 5,
        y=loadings[i, 1] * 5,
        showarrow=True,
        arrowsize=2,
        arrowhead=2,
        xanchor="right",
        yanchor="top"
    )
    fig.add_annotation(
        x=loadings[i, 0] * 5,
        y=loadings[i, 1] * 5,
        ax=0, ay=0,
        xanchor="center",
        yanchor="bottom",
        text=feature,
        yshift=5,
    )
fig.show()

In [24]:
# ['max_l', 'split_l', 'max_r', 's_p', 'f_p', 'f_separation', 'W_A']
a = 'max_r'
pca = PCA(n_components=2)
components = pca.fit_transform(in_out_srgte[[a + '_stdd', 'alpha_stdd', 'beta_stdd', 'vd_stdd', 'sg_stdd', 'D_stdd']])
loadings = pca.components_.T * np.sqrt(pca.explained_variance_)
fig = px.scatter(components, x=0, y=1, color=np.log(in_out_srgte['rho_sat']))
features = [a, 'alpha', 'beta', 'vd', 'sg', 'D']
for i, feature in enumerate(features):
    fig.add_annotation(
        ax=0, ay=0,
        axref="x", ayref="y",
        x=loadings[i, 0] * 5,
        y=loadings[i, 1] * 5,
        showarrow=True,
        arrowsize=2,
        arrowhead=2,
        xanchor="right",
        yanchor="top"
    )
    fig.add_annotation(
        x=loadings[i, 0] * 5,
        y=loadings[i, 1] * 5,
        ax=0, ay=0,
        xanchor="center",
        yanchor="bottom",
        text=feature,
        yshift=5,
    )
fig.show()